# ISIC Occupation Mapping Pipeline
## LLM-based classification of occupations to ISIC Rev.4 economic activities

**What this notebook does:**
1. Loads an ISIC CSV (`Code`, `Title`, `Explanatory Notes`) and Occupations CSV from S3 or local
2. For each occupation sends **all ISIC categories in one prompt** to a reasoning LLM
3. LLM assigns **Grade A** (primary), **B** (secondary), **C** (tertiary) and probability 0-100 for every other ISIC code
4. Saves results as wide CSV, long CSV, Excel, and raw JSON

| Model | Provider | Context | Speed | Cost |
|---|---|---|---|---|
| `deepseek/deepseek-r1:free` | OpenRouter | 128K | Medium | Free |
| `qwen-qwq-32b` | Groq | 131K | ~400 tok/s | Free |

**Theoretical basis:** Role prompting (Brown et al. 2020), Chain-of-Thought (Wei et al. 2022),
structured JSON output (Ouyang et al. 2022), explicit rubric (Zheng et al. 2023 MT-Bench).


In [ ]:
import subprocess, sys
pkgs = ['openai', 'groq', 'pandas', 'tqdm', 'boto3', 'openpyxl']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print("All packages installed.")


In [ ]:
import os, json, time, re, logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import boto3
from tqdm import tqdm
from openai import OpenAI
from groq import Groq

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)
print("Imports OK")


## Step 1 -- Get Your API Keys

### Option A: OpenRouter (DeepSeek R1 -- Best Reasoning, Free)
1. Go to https://openrouter.ai/keys -- sign up, click Create Key
2. Copy key (starts with `sk-or-v1-...`) and paste as `OPENROUTER_API_KEY` below

**Model:** `deepseek/deepseek-r1:free`  
**Cost:** $0/M tokens  
**Also:** DeepSeek's own platform gives 5M free trial tokens: https://platform.deepseek.com/

---

### Option B: Groq (QwQ-32B -- Fastest Free, ~400 tok/s)
1. Go to https://console.groq.com/keys -- sign up (no credit card)
2. Copy key (starts with `gsk_...`) and paste as `GROQ_API_KEY` below

**Free limits:** 30 RPM, ~14,400 RPD

---

**Recommendation:** Get both keys, set `PRIMARY_MODEL = 'both'` -- the pipeline
tries DeepSeek R1 first and falls back to QwQ-32B automatically.


In [ ]:
# ============================================================
#              PASTE YOUR API KEYS HERE
# ============================================================

OPENROUTER_API_KEY = 'sk-or-v1-YOUR_KEY_HERE'   # https://openrouter.ai/keys
GROQ_API_KEY       = 'gsk_YOUR_KEY_HERE'          # https://console.groq.com/keys

# MODEL SELECTION
# 'deepseek' -> DeepSeek R1 via OpenRouter only
# 'groq'     -> Qwen QwQ-32B via Groq only
# 'both'     -> DeepSeek first, auto-fallback to Groq (recommended)
PRIMARY_MODEL = 'both'

# ============================================================
#              AWS / S3 SETTINGS
# ============================================================

USE_S3           = True
S3_BUCKET        = 'YOUR-BUCKET-NAME'
S3_INPUT_PREFIX  = 'isic-mapping/input/'   # trailing slash required
S3_OUTPUT_PREFIX = 'isic-mapping/output/'

# CSV file names (must exist in S3_INPUT_PREFIX when USE_S3=True)
ISIC_CSV_FILENAME        = 'isic_activities.csv'   # Columns: Code, Title, Explanatory Notes
OCCUPATIONS_CSV_FILENAME = 'occupations.csv'        # Columns: Occupation Name, Description

# LOCAL PATHS (used when USE_S3=False)
LOCAL_INPUT_DIR  = './input/'
LOCAL_OUTPUT_DIR = './output/'

# ============================================================
#              PROCESSING SETTINGS
# ============================================================

CHECKPOINT_EVERY = 5     # Save progress every N occupations
MAX_RETRIES      = 3     # Retries per occupation on API error
MAX_NOTE_CHARS   = 600   # Max chars of ISIC explanatory note per category
OPENROUTER_SLEEP = 3.0   # Seconds between OpenRouter calls (free tier is slow)
GROQ_SLEEP       = 2.0   # Seconds between Groq calls

print("Configuration loaded.")
print(f"  Primary model : {PRIMARY_MODEL}")
print(f"  Using S3      : {USE_S3}")


In [ ]:
def _load_s3(bucket, key):
    obj = boto3.client('s3').get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

def _normalise_columns(df, mapping):
    df.columns = df.columns.str.strip()
    rmap = {}
    for std, variants in mapping.items():
        for col in df.columns:
            if col.lower() in [v.lower() for v in variants]:
                rmap[col] = std; break
    return df.rename(columns=rmap)

def load_data():
    if USE_S3:
        isic_df = _load_s3(S3_BUCKET, S3_INPUT_PREFIX + ISIC_CSV_FILENAME)
        occ_df  = _load_s3(S3_BUCKET, S3_INPUT_PREFIX + OCCUPATIONS_CSV_FILENAME)
    else:
        isic_df = pd.read_csv(LOCAL_INPUT_DIR + ISIC_CSV_FILENAME)
        occ_df  = pd.read_csv(LOCAL_INPUT_DIR + OCCUPATIONS_CSV_FILENAME)

    isic_df = _normalise_columns(isic_df, {
        'code':  ['code','isic_code','isic code','Code','Activity Code'],
        'title': ['title','activity title','Title','Activity Title'],
        'explanatory_notes': ['explanatory notes','explanatory_notes',
                               'notes','Explanatory Notes'],
    })
    occ_df = _normalise_columns(occ_df, {
        'occupation_name': ['occupation name','occupation_name',
                             'name','Occupation Name','Occupation'],
        'description':     ['description','occupation description',
                             'job description','Description'],
    })
    logger.info(f'Loaded {len(isic_df)} ISIC, {len(occ_df)} occupations')
    return isic_df, occ_df

isic_df, occupations_df = load_data()

assert 'code'  in isic_df.columns, 'ISIC CSV must have a Code column'
assert 'title' in isic_df.columns, 'ISIC CSV must have a Title column'
assert 'occupation_name' in occupations_df.columns, 'Occupations CSV must have Occupation Name'

print("ISIC cols:", isic_df.columns.tolist())
print("Occ cols: ", occupations_df.columns.tolist())
display(isic_df.head(3))
display(occupations_df.head(3))


In [ ]:
def build_isic_context(df, max_chars=MAX_NOTE_CHARS):
    blocks = []
    for _, row in df.iterrows():
        c = str(row['code']).strip()
        t = str(row['title']).strip()
        n = str(row.get('explanatory_notes', '')).strip()
        if n.lower() == 'nan': n = ''
        if len(n) > max_chars: n = n[:max_chars] + '...'
        b = f'[{c}] {t}'
        if n: b += f'\n    Note: {n}'
        blocks.append(b)
    return '\n\n'.join(blocks)

ISIC_CONTEXT = build_isic_context(isic_df)
ISIC_CODES   = [str(r['code']).strip() for _, r in isic_df.iterrows()]

est = len(ISIC_CONTEXT) // 4
print(f"ISIC context: {len(ISIC_CONTEXT):,} chars (~{est:,} tokens)")
print(f"ISIC codes  : {len(ISIC_CODES)} entries")
if est > 100_000:
    print('WARNING: context large -- reduce MAX_NOTE_CHARS if hitting limits')
else:
    print('Fits within 128K context window OK')


## Prompt Engineering Design

| Technique | Source | Purpose |
|---|---|---|
| Role-based prompting | Brown et al. (2020) GPT-3 | Expert role primes ISIC domain reasoning |
| Chain-of-Thought | Wei et al. (2022) | 4-step decomposition improves ranking by 10-40% |
| Structured JSON schema | Ouyang et al. (2022) | Prevents hallucinated codes, forces completeness |
| Explicit rubric (0/30/70/100) | Zheng et al. (2023) MT-Bench | Makes probability scores consistent across occupations |
| Critical-distinction framing | Turpin et al. (2023) | Separates 'can appear' from 'primarily lives here' |

### One call per occupation (not per occupation x ISIC pair)
Sending all ISIC categories in one prompt enables **cross-ISIC comparative reasoning**: the model
ranks sectors relative to each other rather than scoring each in isolation -- methodologically
superior for ordinal ranking (Ouyang et al., 2022). With 128K context and ~250 tokens per ISIC
entry, up to ~500 ISIC entries fit comfortably.


In [ ]:
SYSTEM_PROMPT = (
    'You are an expert economist and occupational analyst specialising in the '
    'International Standard Industrial Classification of All Economic Activities (ISIC) Rev. 4, '
    'published by the United Nations Statistics Division.\n\n'
    'Your task: identify which ISIC economic activity categories are the PRIMARY home '
    'of a given occupation in the labour market.\n\n'
    'CORE DISTINCTION: You are NOT asking whether an occupation CAN exist in a sector. '
    'You are asking WHERE workers with this occupation PRIMARILY create economic value. '
    'A Mining Safety Inspector primarily exists in ISIC B (Mining and Quarrying) even though '
    'safety inspectors appear in many sectors. An Accountant is cross-cutting with no '
    'single dominant sector.\n\n'
    'Output ONLY valid JSON. No text outside the JSON object.'
)

def build_user_prompt(occ_name, occ_desc, isic_context, isic_codes):
    n         = len(isic_codes)
    codes_str = ', '.join(f'"' + c + '"' for c in isic_codes)
    prob_ph   = ',\n    '.join(f'"' + c + '": <0-100>' for c in isic_codes)

    section = lambda title, body: title + '\n' + body

    output_template = (
        '{\n'
        '  "reasoning": {\n'
        '    "occupation_core_function": "<one sentence: what economic function?>",\n'
        '    "primary_sector_logic": "<why Grade A is the correct primary sector>",\n'
        '    "cross_sector_notes": "<is this cross-cutting? what does that mean?>",\n'
        '    "confidence": "high | medium | low"\n'
        '  },\n'
        '  "top_matches": {\n'
        '    "A": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" },\n'
        '    "B": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" },\n'
        '    "C": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" }\n'
        '  },\n'
        '  "probability_scores": {\n'
        f'    {prob_ph}\n'
        '  }\n'
        '}'
    )

    parts = [
        '## OCCUPATION TO CLASSIFY',
        '',
        f'Name        : {occ_name}',
        f'Description : {occ_desc}',
        '',
        '---',
        f'## ALL ISIC ECONOMIC ACTIVITIES ({n} categories)',
        '',
        isic_context,
        '',
        '---',
        '## CLASSIFICATION TASK',
        '',
        'Work through these steps before producing your output.',
        '',
        'STEP 1 -- OCCUPATION CORE ANALYSIS',
        '  What is the fundamental economic function of this occupation?',
        '  Which organisations employ it as part of their PRIMARY (not support) workforce?',
        '',
        'STEP 2 -- GRADE A (Primary sector)',
        '  The ONE ISIC category that is the definitive home of this occupation:',
        '  - Plurality of workers concentrated here',
        '  - This sector core product/service requires this occupation',
        '  - Workers identify with working in this economic activity',
        '',
        'STEP 3 -- GRADE B (Secondary) and GRADE C (Tertiary)',
        '  Second and third strongest sectors.',
        '',
        f'STEP 4 -- PROBABILITY SCORES FOR ALL {n} ISIC CODES',
        '  Assign a score 0-100 to EVERY code including A/B/C:',
        '  A=100, B=85, C=70 by convention. For others:',
        '  90-100 = near-certain, core to sector',
        '  70-89  = high, many workers expected',
        '  50-69  = moderate, meaningful but not primary',
        '  30-49  = low-moderate, occasional workers',
        '  10-29  = low, rare but plausible',
        '   0-9   = very rare or none',
        '',
        'WARNINGS:',
        '- Do NOT award Grade A to generic support functions (HR, IT, Finance, Legal)',
        '  unless explicitly sector-specific (e.g. Petroleum Finance Analyst).',
        '- For cross-cutting occupations (Driver, Cleaner, Accountant, Security Guard)',
        '  note this in reasoning and give distributed moderate scores.',
        '',
        '---',
        '## OUTPUT -- return ONLY this JSON (no text outside the braces):',
        '',
        output_template,
        '',
        f'probability_scores MUST include all {n} codes: {codes_str}',
        'Set A code = 100, B code = 85, C code = 70. All others: 0-89.',
    ]
    return '\n'.join(parts)

# Quick test
_p = build_user_prompt('Mining Engineer',
    'Designs extraction of minerals from underground mines',
    '[B05] Coal Mining\n    Note: Extraction of coal',
    ['B05', 'C24', 'M71'])
print(_p[:600])
print("...")


In [ ]:
openrouter_client = None
groq_client       = None

if OPENROUTER_API_KEY and not OPENROUTER_API_KEY.endswith('_HERE'):
    openrouter_client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=OPENROUTER_API_KEY,
        default_headers={
            'HTTP-Referer': 'https://github.com/isic-occupation-mapping',
            'X-Title': 'ISIC Occupation Mapping',
        },
    )
    logger.info('OpenRouter client ready (DeepSeek R1)')
else:
    logger.warning('OpenRouter key not set')

if GROQ_API_KEY and not GROQ_API_KEY.endswith('_HERE'):
    groq_client = Groq(api_key=GROQ_API_KEY)
    logger.info('Groq client ready (QwQ-32B)')
else:
    logger.warning('Groq key not set')

if not openrouter_client and not groq_client:
    raise RuntimeError('No LLM client initialised. Please paste at least one API key.')

print(f'Clients: OpenRouter={openrouter_client is not None} | Groq={groq_client is not None}')


In [ ]:
class _RL:
    def __init__(self, gap):
        self.gap = gap; self._last = 0.0
    def wait(self):
        e = time.time() - self._last
        if e < self.gap: time.sleep(self.gap - e)
        self._last = time.time()

_rl_or   = _RL(OPENROUTER_SLEEP)
_rl_groq = _RL(GROQ_SLEEP)

def _call_deepseek(sp, up, temperature=0.1):
    _rl_or.wait()
    r = openrouter_client.chat.completions.create(
        model='deepseek/deepseek-r1:free',
        messages=[{'role':'system','content':sp},{'role':'user','content':up}],
        temperature=temperature, max_tokens=4096)
    return r.choices[0].message.content

def _call_qwq(sp, up, temperature=0.1):
    _rl_groq.wait()
    r = groq_client.chat.completions.create(
        model='qwen-qwq-32b',
        messages=[{'role':'system','content':sp},{'role':'user','content':up}],
        temperature=temperature, max_tokens=4096)
    return r.choices[0].message.content

def call_llm(sp, up, max_retries=MAX_RETRIES):
    if PRIMARY_MODEL == 'deepseek' and openrouter_client:
        queue = [('deepseek-r1', _call_deepseek)]
    elif PRIMARY_MODEL == 'groq' and groq_client:
        queue = [('qwen-qwq-32b', _call_qwq)]
    else:
        queue = []
        if openrouter_client: queue.append(('deepseek-r1', _call_deepseek))
        if groq_client:       queue.append(('qwen-qwq-32b', _call_qwq))

    last_err = None
    for mname, fn in queue:
        for attempt in range(max_retries):
            try:
                t = fn(sp, up)
                if t: return t, mname
            except Exception as e:
                last_err = e
                es = str(e).lower()
                w  = (2**attempt) * (10 if '429' in str(e) or 'rate' in es else 4)
                logger.warning(f'{mname} attempt {attempt+1}: {e} (sleep {w}s)')
                time.sleep(w)
        logger.warning(f'All {max_retries} attempts for {mname} exhausted')
    raise RuntimeError(f'All LLM calls failed. Last: {last_err}')


In [ ]:
def _strip_think(t):
    return re.sub(r'<think>[\s\S]*?</think>', '', t, flags=re.IGNORECASE).strip()

def _extract_json(text):
    text = _strip_think(text)
    try: return json.loads(text)
    except: pass
    for p in [r'```json\s*([\s\S]+?)\s*```', r'```([\s\S]+?)```']:
        m = re.search(p, text)
        if m:
            try: return json.loads(m.group(1).strip())
            except: pass
    # Brace-matching scan
    depth = start = None
    depth = 0
    for i, ch in enumerate(text):
        if ch == '{':
            depth += 1
            if start is None: start = i
        elif ch == '}': depth -= 1
        if depth == 0 and start is not None:
            try: return json.loads(text[start:i+1])
            except: start = None; depth = 0
    return None

def _validate(result, isic_codes):
    if not isinstance(result, dict): return None
    result.setdefault('reasoning', {})
    result.setdefault('top_matches', {})
    result.setdefault('probability_scores', {})
    for g, default_score in [('A',100),('B',85),('C',70)]:
        m = result['top_matches'].setdefault(g, {})
        m.setdefault('isic_code', 'UNKNOWN')
        m.setdefault('title', '')
        m.setdefault('justification', '')
        result['probability_scores'][str(m['isic_code']).strip()] = default_score
    for c in isic_codes:
        result['probability_scores'].setdefault(c, 0)
    for c in list(result['probability_scores']):
        try: result['probability_scores'][c] = max(0,min(100,int(float(result['probability_scores'][c]))))
        except: result['probability_scores'][c] = 0
    return result


In [ ]:
def classify_occupation(occ_name, occ_desc, isic_context, isic_codes,
                         max_retries=MAX_RETRIES):
    user_p = build_user_prompt(occ_name, occ_desc, isic_context, isic_codes)
    for attempt in range(max_retries):
        raw = None
        try:
            raw, model_used = call_llm(SYSTEM_PROMPT, user_p)
            result = _extract_json(raw)
            result = _validate(result, isic_codes)
            if result:
                result['_model']   = model_used
                result['_raw_len'] = len(raw)
                return result
            logger.warning(f'JSON parse failed attempt {attempt+1}')
            time.sleep(3)
        except RuntimeError: raise
        except Exception as e:
            logger.error(f'classify error attempt {attempt+1}: {e}')
            time.sleep(5)
    return {
        'reasoning': {'error': 'failed after all retries'},
        'top_matches': {g: {'isic_code':'ERROR','title':'','justification':''} for g in 'ABC'},
        'probability_scores': {c: 0 for c in isic_codes},
        '_model': 'none',
        '_error': str(raw)[:300] if raw else 'no response',
    }


In [ ]:
CKPT_DIR  = Path('./checkpoints')
CKPT_FILE = CKPT_DIR / 'isic_mapping_progress.json'
CKPT_DIR.mkdir(exist_ok=True)

def save_checkpoint(results):
    with open(CKPT_FILE, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    logger.info(f'Checkpoint saved ({len(results)} records)')

def load_checkpoint():
    if CKPT_FILE.exists():
        with open(CKPT_FILE) as f: data = json.load(f)
        logger.info(f'Resumed: {len(data)} previously processed')
        return data
    return []


In [ ]:
def run_pipeline(occupations_df, isic_context, isic_codes, resume=True):
    results = load_checkpoint() if resume else []
    done    = {r['occupation_name'] for r in results if 'occupation_name' in r}
    todo    = occupations_df[~occupations_df['occupation_name'].isin(done)].copy()

    logger.info(f'Total:{len(occupations_df)}  Done:{len(done)}  Remaining:{len(todo)}')

    for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo), desc='Classifying')):
        name = str(row['occupation_name']).strip()
        desc = str(row.get('description', '')).strip()
        logger.info(f'[{i+1}/{len(todo)}] {name}')
        try:
            res = classify_occupation(name, desc, isic_context, isic_codes)
        except Exception as e:
            logger.error(f'Fatal error for {name}: {e}')
            res = {'reasoning': {'error': str(e)},
                   'top_matches': {g: {'isic_code':'ERROR','title':'','justification':''} for g in 'ABC'},
                   'probability_scores': {c: 0 for c in isic_codes}, '_model': 'none'}
        res['occupation_name']        = name
        res['occupation_description'] = desc
        res['processed_at']           = datetime.now().isoformat()
        results.append(res)
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(results)

    save_checkpoint(results)
    logger.info(f'Pipeline complete -- {len(results)} total')
    return results

print('=' * 60)
print('STARTING ISIC CLASSIFICATION PIPELINE')
print(f'  Occupations : {len(occupations_df)}')
print(f'  ISIC codes  : {len(ISIC_CODES)}')
print(f'  Model       : {PRIMARY_MODEL}')
print('=' * 60)

ALL_RESULTS = run_pipeline(
    occupations_df = occupations_df,
    isic_context   = ISIC_CONTEXT,
    isic_codes     = ISIC_CODES,
    resume         = True,   # set False to restart from scratch
)
print(f'Done! {len(ALL_RESULTS)} occupations classified.')


In [ ]:
def to_wide_df(results, isic_codes):
    rows = []
    for r in results:
        row = {
            'Occupation_Name':        r.get('occupation_name', ''),
            'Occupation_Description': r.get('occupation_description', ''),
            'Model_Used':             r.get('_model', ''),
            'Processed_At':           r.get('processed_at', ''),
            'Confidence':             r.get('reasoning', {}).get('confidence', ''),
            'Core_Function':          r.get('reasoning', {}).get('occupation_core_function', ''),
            'Primary_Sector_Logic':   r.get('reasoning', {}).get('primary_sector_logic', ''),
            'Cross_Sector_Notes':     r.get('reasoning', {}).get('cross_sector_notes', ''),
        }
        for g in ['A', 'B', 'C']:
            m = r.get('top_matches', {}).get(g, {})
            row[f'Grade_{g}_Code']          = m.get('isic_code', '')
            row[f'Grade_{g}_Title']         = m.get('title', '')
            row[f'Grade_{g}_Justification'] = m.get('justification', '')
        ps = r.get('probability_scores', {})
        for c in isic_codes:
            row[f'Prob_{c}'] = ps.get(c, ps.get(str(c), 0))
        rows.append(row)
    return pd.DataFrame(rows)

def to_long_df(results):
    rows = []
    for r in results:
        name = r.get('occupation_name', '')
        gc = {}
        for g in ['A', 'B', 'C']:
            m = r.get('top_matches', {}).get(g, {})
            rows.append({'Occupation_Name': name, 'ISIC_Code': m.get('isic_code',''),
                         'ISIC_Title': m.get('title',''), 'Type': 'Grade', 'Grade': g,
                         'Score': {'A':100,'B':85,'C':70}[g],
                         'Justification': m.get('justification','')})
            gc[m.get('isic_code','')] = True
        for code, score in r.get('probability_scores', {}).items():
            if code not in gc:
                rows.append({'Occupation_Name': name, 'ISIC_Code': code,
                             'ISIC_Title': '', 'Type': 'Probability',
                             'Grade': None, 'Score': score, 'Justification': ''})
    return pd.DataFrame(rows)

results_wide = to_wide_df(ALL_RESULTS, ISIC_CODES)
results_long = to_long_df(ALL_RESULTS)
print(f'Wide: {results_wide.shape}  |  Long: {results_long.shape}')
display(results_wide[['Occupation_Name','Grade_A_Code','Grade_A_Title',
                       'Grade_B_Code','Grade_C_Code','Confidence']].head(5))


In [ ]:
def save_all(results_wide, results_long, all_results):
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = Path('./output'); out.mkdir(exist_ok=True)

    wide_csv = out / f'isic_mapping_wide_{ts}.csv'
    long_csv = out / f'isic_mapping_long_{ts}.csv'
    raw_json = out / f'isic_mapping_raw_{ts}.json'
    xlsx     = out / f'isic_mapping_{ts}.xlsx'

    results_wide.to_csv(wide_csv, index=False)
    results_long.to_csv(long_csv, index=False)
    with open(raw_json, 'w') as f: json.dump(all_results, f, indent=2, default=str)

    with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
        results_wide.to_excel(w, sheet_name='Wide_Format', index=False)
        results_long.to_excel(w, sheet_name='Long_Format', index=False)
        pd.DataFrame({'Metric':['Total','Successful','Errors','Timestamp'],
                      'Value': [len(all_results),
                        sum(1 for r in all_results if r.get('top_matches',{}).get('A',{}).get('isic_code','')!='ERROR'),
                        sum(1 for r in all_results if r.get('top_matches',{}).get('A',{}).get('isic_code','')=='ERROR'),
                        ts]}).to_excel(w, sheet_name='Summary', index=False)

    print(f'Saved: {wide_csv}  {long_csv}  {raw_json}  {xlsx}')
    if USE_S3:
        key = f'{S3_OUTPUT_PREFIX}isic_mapping_{ts}.xlsx'
        boto3.client('s3').upload_file(str(xlsx), S3_BUCKET, key)
        print(f'Uploaded to s3://{S3_BUCKET}/{key}')
    return str(xlsx)

excel_path = save_all(results_wide, results_long, ALL_RESULTS)
print('All outputs saved.')


In [ ]:
print('=' * 60)
print('CLASSIFICATION SUMMARY')
print('=' * 60)
total  = len(ALL_RESULTS)
errors = sum(1 for r in ALL_RESULTS
             if r.get('top_matches',{}).get('A',{}).get('isic_code','')=='ERROR')
print(f'Total: {total}  Successful: {total-errors}  Errors: {errors}')

if not results_wide.empty:
    title_map = dict(zip(isic_df['code'].astype(str).str.strip(), isic_df['title']))
    print('\n-- Top 10 Grade-A ISIC Sectors --')
    for code, cnt in results_wide['Grade_A_Code'].value_counts().head(10).items():
        print(f'  [{code}] {title_map.get(str(code).strip(),"Unknown"):<55} -> {cnt} occupations')
    if 'Confidence' in results_wide.columns:
        print('\n-- Confidence Distribution --')
        print(results_wide['Confidence'].value_counts().to_string())
    print('\n-- Models Used --')
    print(results_wide['Model_Used'].value_counts().to_string())


## How to Interpret the Output

### Wide Format (`isic_mapping_wide_*.csv`) -- one row per occupation

| Column | Meaning |
|---|---|
| `Grade_A_Code` / `Grade_A_Title` | Primary ISIC sector (plurality of workers here) |
| `Grade_B_Code` / `Grade_C_Code` | Second and third strongest sectors |
| `Grade_*_Justification` | LLM reasoning for the assignment |
| `Prob_{ISIC_CODE}` | 0-100 probability for every ISIC code |
| `Confidence` | `high` / `medium` / `low` -- model self-assessed certainty |
| `Cross_Sector_Notes` | Flags cross-cutting occupations |

### Long Format (`isic_mapping_long_*.csv`) -- one row per (occupation, ISIC code)
Better for heat-maps, pivot analysis, filtering by sector, Power BI / Tableau.

### Caveats
1. **Cross-cutting occupations** (Accountant, Driver, Cleaner): Grade A is less meaningful;
   use `Prob_*` columns as the primary signal.
2. **ISIC granularity**: 4-digit Classes (~420 codes) push prompts toward 100K tokens;
   reduce `MAX_NOTE_CHARS` or work at 2-digit Division level if needed.
3. **Free-tier rate limits**: Groq 30 RPM is the binding constraint; at 2s sleep you get
   ~30 occupations/min and ~1,800/hour.
4. **Hallucinated codes**: The validator flags codes not in your ISIC list as `UNKNOWN`.
   Inspect rows where `Grade_A_Code == 'UNKNOWN'`.
5. **Reproducibility**: Set `temperature=0.0` in `_call_deepseek` / `_call_qwq` for
   deterministic output at the cost of slightly lower reasoning quality.
